In [1]:
!pip install pandas numpy scikit-learn xgboost joblib matplotlib

In [2]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("UAVIDS-2025.csv")

print("Shape:", df.shape)
df.head()

Shape: (122171, 23)


,FlowID,FlowDuration/s,SrcAddr,SrcPort,DstAddr,DstPort,Protocol,TxPackets,RxPackets,LostPackets,...,RxPacketRate/s,TxByteRate/s,RxByteRate/s,MeanDelay/s,MeanJitter/s,Throughput/Kbps,MeanPacketSize,PacketDropRate,AverageHopCount,label
0,1,175.007,192.168.0.150,9,192.168.0.16,9,UDP,251,215,36,...,1.22853,109.002,93.3679,0.022152,0.024406,0.746943,76,0.143426,0.074419,Sybil Attack
1,2,175.008,192.168.0.150,9,192.168.0.19,9,UDP,251,223,28,...,1.27423,109.001,96.8411,0.019637,0.018678,0.774729,76,0.111554,0.053812,Sybil Attack
2,3,175.010,192.168.0.150,9,192.168.0.24,9,UDP,251,224,27,...,1.27993,108.999,97.2744,0.023701,0.023917,0.778195,76,0.107570,0.044643,Sybil Attack
3,4,175.014,192.168.0.150,9,192.168.0.34,9,UDP,251,212,39,...,1.21133,108.997,92.0614,0.034014,0.025408,0.736491,76,0.155378,0.033019,Sybil Attack
4,5,175.017,192.168.0.150,9,192.168.0.39,9,UDP,251,215,36,...,1.22845,108.995,93.3623,0.034396,0.024933,0.746899,76,0.143426,0.009302,Sybil Attack


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nClass Distribution:")
print(df["label"].value_counts())

Columns:
['FlowID', 'FlowDuration/s', 'SrcAddr', 'SrcPort', 'DstAddr', 'DstPort', 'Protocol', 'TxPackets', 'RxPackets', 'LostPackets', 'TxBytes', 'RxBytes', 'TxPacketRate/s', 'RxPacketRate/s', 'TxByteRate/s', 'RxByteRate/s', 'MeanDelay/s', 'MeanJitter/s', 'Throughput/Kbps', 'MeanPacketSize', 'PacketDropRate', 'AverageHopCount', 'label']

Dataset Shape:
(122171, 23)

Missing Values:
FlowID             0
FlowDuration/s     0
SrcAddr            0
SrcPort            0
DstAddr            0
DstPort            0
Protocol           0
TxPackets          0
RxPackets          0
LostPackets        0
TxBytes            0
RxBytes            0
TxPacketRate/s     0
RxPacketRate/s     0
TxByteRate/s       0
RxByteRate/s       0
MeanDelay/s        0
MeanJitter/s       0
Throughput/Kbps    0
MeanPacketSize     0
PacketDropRate     0
AverageHopCount    0
label              0
dtype: int64

Duplicate Rows:
0

Class Distribution:
label
Normal Traffic      26172
Blackhole Attack    26110
Wormhole Attack     2

In [5]:
df = df.drop(["Protocol", "FlowID"], axis=1)

print(df.shape)

(122171, 21)


In [6]:
src_encoder = LabelEncoder()
dst_encoder = LabelEncoder()
label_encoder = LabelEncoder()

df["SrcAddr"] = src_encoder.fit_transform(df["SrcAddr"])
df["DstAddr"] = dst_encoder.fit_transform(df["DstAddr"])
df["label"] = label_encoder.fit_transform(df["label"])

print("Label Mapping:")
for i, attack in enumerate(label_encoder.classes_):
    print(i, "=", attack)

Label Mapping:
0 = Blackhole Attack
1 = Flooding Attack
2 = Normal Traffic
3 = Sybil Attack
4 = Wormhole Attack


In [7]:
df_clean = df.drop(["SrcAddr", "DstAddr"], axis=1)

print(df_clean.shape)

(122171, 19)


In [8]:
X = df_clean.drop("label", axis=1)
y = df_clean["label"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (122171, 18)
y Shape: (122171,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (97736, 18)
Test : (24435, 18)


In [10]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

xgb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6, 8],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        random_state=42,
        # Removed eval_metric to avoid serialization issues
        # Removed tree_method to use default settings
        n_jobs=1                 # Keep single-threaded for XGBClassifier
    ),
    param_grid=xgb_params,
    cv=5,
    scoring="accuracy",
    n_jobs=1,                    # Changed: Use single process to avoid serialization issues
    verbose=2
)

xgb_grid.fit(X_train, y_train)

print("Best Parameters:")
print(xgb_grid.best_params_)

print("\nBest CV Accuracy:")
print(xgb_grid.best_score_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=0.8; total time=   5.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   3.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   3.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=100, subsample=1.0; total time=   3.5s
[CV] END c

In [11]:
best_xgb = xgb_grid.best_estimator_

y_pred = best_xgb.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.956292203806016
              precision    recall  f1-score   support

           0       0.95      0.86      0.91      5222
           1       1.00      0.99      0.99      3945
           2       0.99      0.99      0.99      5235
           3       0.99      1.00      1.00      4816
           4       0.87      0.95      0.91      5217

    accuracy                           0.96     24435
   macro avg       0.96      0.96      0.96     24435
weighted avg       0.96      0.96      0.96     24435



In [12]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_xgb.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
15,MeanPacketSize,0.285551
10,TxByteRate/s,0.204738
1,SrcPort,0.122638
16,PacketDropRate,0.089419
6,TxBytes,0.085326
3,TxPackets,0.046716
0,FlowDuration/s,0.026283
8,TxPacketRate/s,0.026220
7,RxBytes,0.022336
5,LostPackets,0.020554


In [13]:
top_features = feature_importance.head(10)["Feature"].tolist()

print(top_features)

['MeanPacketSize', 'TxByteRate/s', 'SrcPort', 'PacketDropRate', 'TxBytes', 'TxPackets', 'FlowDuration/s', 'TxPacketRate/s', 'RxBytes', 'LostPackets']


In [14]:
X_top = X[top_features]

X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [15]:
final_xgb = XGBClassifier(
    n_estimators=best_xgb.n_estimators,
    learning_rate=best_xgb.learning_rate,
    max_depth=best_xgb.max_depth,
    subsample=best_xgb.subsample,
    colsample_bytree=best_xgb.colsample_bytree,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1
)

final_xgb.fit(X_train_top, y_train_top)

print("Training Accuracy :", final_xgb.score(X_train_top, y_train_top))
print("Testing Accuracy  :", final_xgb.score(X_test_top, y_test_top))

Training Accuracy : 0.9629409838749283
Testing Accuracy  : 0.9515039901780233


In [21]:
import joblib
import sklearn
import xgboost
from datetime import datetime
import hashlib

# Create a hash of the dataset for version tracking
dataset_hash = hashlib.md5(
    pd.util.hash_pandas_object(df, index=True).values
).hexdigest()

artifact = {
    # Model
    "model": final_xgb,

    # Features
    "feature_columns": top_features,
    "feature_dtypes": X_train_top.dtypes.astype(str).to_dict(),

    # Labels
    "label_encoder": label_encoder,
    "class_names": label_encoder.classes_.tolist(),

    # Metadata
    "model_name": "UAV IDS Tuned XGBoost",
    "model_version": "1.0.0",
    "description": "Tuned XGBoost model for UAV Intrusion Detection using top selected features.",

    # Metrics
    "train_accuracy": float(final_xgb.score(X_train_top, y_train_top)),
    "test_accuracy": float(final_xgb.score(X_test_top, y_test_top)),

    # Environment
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "random_state": 42,

    # Dataset info
    "dataset_rows": len(df),
    "dataset_columns": len(df.columns),
    "dataset_hash": dataset_hash,

    # Timestamp
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

joblib.dump(artifact, "uav_ids_xgboost_v1.0.pkl")

print("Production Artifact Created Successfully!")

Production Artifact Created Successfully!


In [23]:
import joblib

artifact = joblib.load("uav_ids_xgboost_v1.0.pkl")

print(artifact.keys())

dict_keys(['model', 'feature_columns', 'feature_dtypes', 'label_encoder', 'class_names', 'model_name', 'model_version', 'description', 'train_accuracy', 'test_accuracy', 'sklearn_version', 'xgboost_version', 'random_state', 'dataset_rows', 'dataset_columns', 'dataset_hash', 'created_at'])


In [18]:
import os

print(os.getcwd())

C:\Users\ASUS\anaconda_projects\b6ebc2ec-03bb-43d2-af67-13189d00d418


In [1]:
from predictor import UAVPredictor

predictor = UAVPredictor("uav_ids_xgboost_v1.0.pkl")

In [4]:
import pandas as pd
import joblib

# Load the original dataset
df = pd.read_csv("UAVIDS-2025.csv")

# Remove the unused columns
df = df.drop(["Protocol", "FlowID"], axis=1)

# Encode the label (same way as training)
artifact = joblib.load("uav_ids_xgboost_v1.0.pkl")
label_encoder = artifact["label_encoder"]

df["label"] = label_encoder.transform(df["label"])

# Keep only the required features
sample = df[artifact["feature_columns"]].iloc[0].to_dict()

print(sample)

{'MeanPacketSize': 76.0, 'TxByteRate/s': 109.002, 'SrcPort': 9.0, 'PacketDropRate': 0.143426, 'TxBytes': 19076.0, 'TxPackets': 251.0, 'FlowDuration/s': 175.007, 'TxPacketRate/s': 1.43423, 'RxBytes': 16340.0, 'LostPackets': 36.0}


In [5]:
from predictor import UAVPredictor

predictor = UAVPredictor("uav_ids_xgboost_v1.0.pkl")

predictor.predict(sample)

{'prediction': 'Sybil Attack', 'confidence': 99.33}